# Proyecto 4 

**Mateo Serrato Ascencio**

**Rodrigo Rosales Díaz**

**María Gamba Santibáñez**


Objetivo: Maximizar la eficiencia Térmica del dispositivo

### FUNDAMENTOS FÍSICOS
- El fluido caliente circula por el tubo interior
- El fluido frío circula en contracorriente por el anillo interior

In [ ]:
import math
import numpy as np
import random

# -------------------CONSTANTES DEL PROYECTO -------------------
# Parámetros físicos 
T_H_IN = 80.0    # Temperatura entrada caliente (°C)
T_C_IN = 20.0    # Temperatura entrada fría (°C)
U = 500.0      # Coeficiente global de transferencia (W/m^2K)
C_P = 4180.0     # Calor específico (J/kgK) [asumiendo cc = ch]

# Límites de las variables de decisión (Restricciones) 
# X = [Di, Do, L, m_h, m_c]
BOUNDS = [
    [0.01, 0.05],   # Di: Diámetro interior 
    [0.015, 0.06],  # Do: Diámetro exterior 
    [1.0, 10.0],    # L: Longitud 
    [0.05, 0.5],    # m_h: Flujo másico caliente 
    [0.05, 0.5]     # m_c: Flujo másico frío (asumido de m_h) 
]

# -------------------FUNCIONES-------------------

# Función Objetivo (Maximizar Eficiencia)
def fun_obj(X):
    # Desempacar el vector de variables
    Di, Do, L, m_h, m_c = X
    
    # 1. Calcular Capacidades Térmicas (Ch, Cc) 
    C_h = m_h * C_P
    C_c = m_c * C_P
    
    # 2. Encontrar Cmin, Cmax y Cr 
    C_min = min(C_h, C_c)
    C_max = max(C_h, C_c)
    
    # Evitar división por cero si C_max es 0 (aunque los límites lo impiden)
    if C_max == 0: return 1e6 # Penalización alta
    
    C_r = C_min / C_max
    
    # 3. Calcular NTU (Number of Transfer Units) 
    A = math.pi * Di * L  # Área de intercambio 
    
    # Evitar división por cero si C_min es 0
    if C_min == 0: return 1e6 # Penalización alta
        
    NTU = (U * A) / C_min
    
    # 4. Calcular Eficiencia (eta) 
    # Caso especial si Cr = 1 (para evitar división por cero)
    if C_r == 1:
        eta = NTU / (1.0 + NTU)
    else:
        num = 1.0 - math.exp(-NTU * (1.0 - C_r))
        den = 1.0 - C_r * math.exp(-NTU * (1.0 - C_r))
        
        # Evitar división por cero si el denominador es 0
        if den == 0: return 1e6 # Penalización alta
        eta = num / den

    # 5. Manejo de Restricciones (Penalización) 
    # Restricción: T_h_out > T_c_in
    # Calcular T_h_out 
    T_h_out = T_H_IN - eta * (C_min / C_h) * (T_H_IN - T_C_IN)
    
    if T_h_out <= T_C_IN:
        # Violación de restricción: penalizar fuertemente
        return 1e6  # Un valor muy alto (malo para minimizar)
    
    # El objetivo es MAXIMIZAR eficiencia 
    # Como el DE está configurado para MINIMIZAR (f2 < f1),
    # retornamos el negativo de la eficiencia.
    return -eta

# Función para inicializar la población
def ini_pob(tam_pob, bounds):
    pob = []
    num_vars = len(bounds)
    for _ in range(tam_pob):
        individuo = []
        for j in range(num_vars):
            # Usar los límites específicos para cada variable
            min_val = bounds[j][0]
            max_val = bounds[j][1]
            individuo.append(random.uniform(min_val, max_val))
        pob.append(individuo)
    return pob

# Función para evaluar la población
def evaluar(pob):
    fitness = []
    n = len(pob)
    for i in range(n):
        # Pasar el individuo completo (vector) a la función objetivo
        f = fun_obj(pob[i])
        fitness.append(f)
    return fitness

# Función para construir los valores mutantes
def mutar(individuo_idx, pob, F):
    n = len(pob)
    num_vars = len(pob[0])
    
    indices = list(range(n))
    indices.remove(individuo_idx) # Quitar el índice actual
    
    # Seleccionar r1, r2, r3 sin repetir y diferentes al actual
    r1, r2, r3 = random.sample(indices, 3)
    
    vector_mutante = []
    for j in range(num_vars):
        mutacion = pob[r1][j] + (F * (pob[r2][j] - pob[r3][j]))
        vector_mutante.append(mutacion)
    return vector_mutante

# Funcion de cruza 
def cruza(vector_mutante, individuo, CR):
    vector_prueba = []
    n = len(individuo)
    j_rand = random.randint(0, n - 1)
    for j in range(n):
        if (random.random() < CR or j == j_rand):
            vector_prueba.append(vector_mutante[j])
        else:
            vector_prueba.append(individuo[j])
    return vector_prueba

# Función para asegurar que el vector esté dentro de los límites
def clip_bounds(vector, bounds):
    clipped_vector = []
    for i in range(len(vector)):
        val = vector[i]
        min_val = bounds[i][0]
        max_val = bounds[i][1]
        
        if val < min_val:
            val = min_val
        elif val > max_val:
            val = max_val
        clipped_vector.append(val)
    return clipped_vector

# FUNCIÓN PARA SELECCIONAR INDIVIDUOS
def seleccion(individuo, vector_prueba):
    # Pasar los vectores completos
    f1 = fun_obj(individuo)
    f2 = fun_obj(vector_prueba)
    
    # Minimización (buscamos el valor más pequeño de -eta)
    if (f2 < f1):
        return vector_prueba
    else:
        return individuo
    
# -------------------IMPLEMENTACIÓN-------------------
# Parámetros del algoritmo 
tam_pob = 30
F = 0.8
CR = 0.7
generaciones = 100

# 1. Inicializar Población
P = ini_pob(tam_pob, BOUNDS)

# 2. Ciclo Evolutivo
for g in range(generaciones):
    nueva_poblacion = []
    for i in range(tam_pob):
        # Mutación
        MUT = mutar(i, P, F)
        
        # Cruza
        VEC_PRUEBA = cruza(MUT, P[i], CR)
        
        # Esto es para que las variables no se salgan de [0.01, 0.05], etc.
        VEC_PRUEBA = clip_bounds(VEC_PRUEBA, BOUNDS)
        
        # Selección
        IND_NUEVO = seleccion(P[i], VEC_PRUEBA)
        nueva_poblacion.append(IND_NUEVO)
        
    P = nueva_poblacion

# -------------------RESULTADOS-------------------
fitness = evaluar(P)
mejor_indice = fitness.index(min(fitness))
mejor_solucion = P[mejor_indice]
mejor_valor_obj = fitness[mejor_indice]

# Como minimizamos -eta, la eficiencia real es -mejor_valor_obj
mejor_eficiencia = -mejor_valor_obj

print("\n--- Optimización Terminada ---")
print(f"Mejor Valor de la función (Minimizando -eta): {mejor_valor_obj:.6f}")
print(f"Mejor Eficiencia (eta) encontrada: {mejor_eficiencia:.6f} (o {mejor_eficiencia*100:.2f}%)")

print("\nMejor Solución (Variables):")
print(f"  Di (Diámetro int): {mejor_solucion[0]:.4f} m")
print(f"  Do (Diámetro ext): {mejor_solucion[1]:.4f} m")
print(f"  L  (Longitud):      {mejor_solucion[2]:.4f} m")
print(f"  m_h (Flujo caliente): {mejor_solucion[3]:.4f} kg/s")
print(f"  m_c (Flujo frío):   {mejor_solucion[4]:.4f} kg/s")


--- Optimización Terminada ---
Mejor Valor de la función (Minimizando -eta): -0.969317
Mejor Eficiencia (eta) encontrada: 0.969317 (o 96.93%)

Mejor Solución (Variables):
  Di (Diámetro int): 0.0500 m
  Do (Diámetro ext): 0.0150 m
  L  (Longitud):      10.0000 m
  m_h (Flujo caliente): 0.5000 kg/s
  m_c (Flujo frío):   0.0500 kg/s
